# Credit Card Customer Segmentation

This notebook develops a reproducible K-means baseline for behavioral customer segmentation. It separates data preparation, model selection, cluster profiling, and business interpretation, while documenting important limitations.

## Business objective

Segment 10,127 credit-card customers using behavioral and relationship attributes so that marketing, product, service, and risk teams can design differentiated strategies. This is an unsupervised problem: there is no target variable to predict.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_csv("Customer_Data.csv")
print(f"Rows: {df.shape[0]:,}; original features: {df.shape[1]}")
print(f"Missing values: {int(df.isna().sum().sum())}; duplicate rows: {int(df.duplicated().sum())}")
df.head()

## Data preparation

The supplied analysis file contains no missing values or duplicate rows, so no imputation or duplicate removal is required. Categorical features are one-hot encoded. All model features are standardized because K-means uses Euclidean distance and is sensitive to scale.

In [ ]:
categorical_columns = df.select_dtypes(include="object").columns.tolist()
X = pd.get_dummies(df, columns=categorical_columns, drop_first=True, dtype=float)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Categorical columns: {categorical_columns}")
print(f"Model features after encoding: {X.shape[1]}")

## Candidate model evaluation

K-means models from 2 through 10 clusters are compared using silhouette score and inertia. Silhouette score is the primary selection metric in this baseline; inertia is shown as a secondary diagnostic. `n_init` is fixed explicitly for reproducibility.

In [ ]:
results = []
models = {}
for k in range(2, 11):
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = model.fit_predict(X_scaled)
    results.append({
        "k": k,
        "silhouette_score": silhouette_score(
            X_scaled, labels, sample_size=3000, random_state=RANDOM_STATE
        ),
        "inertia": model.inertia_,
    })
    models[k] = model

evaluation = pd.DataFrame(results)
evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=evaluation, x="k", y="silhouette_score", marker="o", ax=axes[0])
axes[0].set_title("Silhouette score by number of clusters")
axes[0].set_xticks(evaluation["k"])

sns.lineplot(data=evaluation, x="k", y="inertia", marker="o", ax=axes[1])
axes[1].set_title("Inertia by number of clusters")
axes[1].set_xticks(evaluation["k"])
plt.tight_layout()
plt.savefig("Cluster_Model_Selection.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
best_k = int(evaluation.loc[evaluation["silhouette_score"].idxmax(), "k"])
final_model = models[best_k]
cluster_labels = final_model.labels_
best_score = float(evaluation.loc[evaluation["k"] == best_k, "silhouette_score"].iloc[0])

print(f"Selected k: {best_k}")
print(f"Silhouette score: {best_score:.4f}")

## Cluster visualization

PCA is used only to create a two-dimensional visualization. The K-means model above was fitted on the complete standardized feature set. The explained-variance percentage is reported so the 2D projection is not overinterpreted.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_.sum()

plt.figure(figsize=(9, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap="tab10", alpha=0.45, s=12)
plt.title(f"PCA projection of K-means segments (k={best_k}; 2D variance={explained:.1%})")
plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.colorbar(label="Cluster")
plt.tight_layout()
plt.savefig("Cluster_PCA_Projection.png", dpi=160, bbox_inches="tight")
plt.show()

print(f"Variance explained by the first two components: {explained:.2%}")

## Cluster sizes and profiles

Cluster labels are arbitrary identifiers. Profiles below compare each cluster's numerical-feature means with the overall population. Business names should be assigned only after reviewing the complete profile, not inferred from the label number.

In [ ]:
segmented = df.copy()
segmented["Cluster"] = cluster_labels

cluster_sizes = segmented["Cluster"].value_counts().sort_index().rename("customers")
cluster_shares = (cluster_sizes / len(segmented)).rename("share")
pd.concat([cluster_sizes, cluster_shares], axis=1)

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns
profile_means = segmented.groupby("Cluster")[numeric_columns].mean()
overall_mean = df[numeric_columns].mean()
overall_std = df[numeric_columns].std()
profile_z = profile_means.subtract(overall_mean, axis=1).divide(overall_std, axis=1)

key_features = [
    "Credit_Limit", "Total_Revolving_Bal", "Avg_Open_To_Buy",
    "Total_Trans_Amt", "Total_Trans_Ct", "Avg_Utilization_Ratio",
    "Total_Relationship_Count", "Months_Inactive_12_mon",
    "Contacts_Count_12_mon"
]
profile_means[key_features].round(2)

In [ ]:
plt.figure(figsize=(12, 5))
sns.heatmap(profile_z[key_features], cmap="vlag", center=0, annot=True, fmt=".2f")
plt.title("Cluster profiles: standard deviations from the population mean")
plt.xlabel("Feature")
plt.ylabel("Cluster")
plt.tight_layout()
plt.savefig("Cluster_Profile_Heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

## Business interpretation and next steps

The segmentation is a baseline rather than a production-ready decision system. The best silhouette score is low, indicating overlapping segments. Before activation, the analysis should:

1. Review outliers and redundant features.
2. Consider weighting behavioral variables more heavily than demographics.
3. Compare K-means with hierarchical clustering, Gaussian mixtures, or methods designed for mixed data.
4. Test stability across samples and time periods.
5. Have domain experts validate whether the profiles are understandable and actionable.
6. Run controlled campaigns and measure incremental conversion, retention, revenue, and profitability.

In [ ]:
segmented.to_csv("Segmented_Customers.csv", index=False)
evaluation.to_csv("Cluster_Evaluation.csv", index=False)
profile_means.to_csv("Cluster_Profiles.csv")
print("Saved Segmented_Customers.csv, Cluster_Evaluation.csv, and Cluster_Profiles.csv")